# Diagnóstico de aprendizagem e lacunas por questionário

## Problema
Professores do Ensino Fundamental I (1º ao 5º ano) não têm um jeito rápido e
padronizado de saber **em quais conteúdos** cada aluno está abaixo do esperado
para a sua série, nem **o que fazer** a respeito.

## Referências
- **BNCC:** define o que se espera que o aluno aprenda em cada ano.
- **SAEB:** avalia Língua Portuguesa e Matemática; por isso o projeto começa por essas duas disciplinas.
- **Compromisso Nacional Criança Alfabetizada (MEC):** a leitura nos anos iniciais continua sendo um dos conteúdos avaliados.

## Quem se beneficia
- **O aluno**, que recebe reforço no conteúdo certo antes de acumular atraso.
- **O professor**, que sabe o que reforçar com cada aluno e com a turma.

## O que a IA faz
1. O aluno responde a um questionário da sua série: **2 disciplinas × 4 conteúdos × 3 questões**.
2. O sistema corrige as respostas e conta os acertos em cada conteúdo.
3. Um modelo de Machine Learning estima, **para cada conteúdo**, a probabilidade de
   haver uma **lacuna** (desempenho abaixo do esperado para a série).
   Trata-se de uma classificação **multirrótulo**: um aluno pode ter nenhuma, uma ou várias lacunas.
4. A partir das lacunas, o sistema gera os **insights**:
   - o **nível em cada disciplina**;
   - os **conteúdos a reforçar**, do mais urgente ao menos urgente;
   - uma **estratégia de estudo** para cada conteúdo, tirada de um catálogo fixo e revisável.

```
respostas ──► validar ──► corrigir ──► 13 features ──► modelo ──► 8 probabilidades
 (série +      (inválido    (acertos       (acertos +      (regressão     de lacuna
  24 letras)    para aqui)   por conteúdo)  série)          logística)        │
                                                                              ▼
     insights ◄── nível por disciplina + conteúdos a reforçar ◄── lacuna / sem_lacuna / a_confirmar
```

Por que o modelo e não só uma regra pelos acertos (como "errou alguma questão do conteúdo")?
Com apenas 3 questões, um chute certo ou um erro de distração mudam o resultado. O modelo usa
também o desempenho nos **outros** conteúdos e a **série** para decidir com mais segurança.
O ganho sobre a melhor regra desse tipo é medido nos passos 9 e 13.

## Conceitos básicos
- **Machine Learning supervisionado:** o modelo aprende a partir de exemplos já rotulados
  (aqui, alunos cujas lacunas o professor já avaliou) e depois prevê o rótulo de alunos novos.
- **Feature:** cada informação de entrada do modelo (aqui, os acertos em cada conteúdo e a série).
- **Rótulo (ou alvo):** o que se quer prever (aqui, lacuna sim/não em cada conteúdo).
- **Classificação binária × multirrótulo:** binária é responder uma pergunta sim/não; multirrótulo
  é responder várias perguntas sim/não ao mesmo tempo para o mesmo aluno (8, uma por conteúdo).
- **Treino e teste:** o modelo aprende com uma parte dos dados (treino) e é avaliado numa parte que
  nunca viu (teste), para medir como ele se sairia com alunos novos.

## Qual erro é o mais grave
**Não detectar uma lacuna que existe:** o aluno ficaria sem o reforço de que precisa.
Apontar uma lacuna que não existe custa menos: uma atividade extra de revisão.

**Métricas** (calculadas por conteúdo e depois com média entre os 8 conteúdos):
- **Prioritária:** recall de lacuna (de todas as lacunas reais, quantas o modelo encontra).
- **De equilíbrio:** F1-macro. Impede que o modelo "ganhe" no recall simplesmente
  apontando lacuna em tudo.

## Uso responsável
- O resultado é uma **triagem**: apoia o professor, não substitui a avaliação dele.
- Questionários inválidos **não recebem diagnóstico**, e conteúdos com resultado
  duvidoso aparecem como **"a confirmar pelo professor"** (falhar fechado).
- As estratégias de estudo vêm de um **catálogo fixo**, não são geradas por IA no momento
  do uso: o texto pode ser revisado por pedagogos e o sistema funciona offline.
- Dados de crianças são sensíveis (LGPD, art. 14). Este projeto usa apenas **dados simulados**.

## ODS
- ODS 4 (Educação de qualidade)
- ODS 10 (Redução das desigualdades)

## Como executar
- **Python 3.11 ou superior** (exigência do pandas 3).
- Instale as dependências: `pip install -r requirements.txt` (de preferência dentro de um ambiente virtual, `python -m venv .venv`).
- Rode `Kernel → Restart & Run All`. As células dependem umas das outras e precisam rodar **em ordem**, de cima para baixo.
- A semente fixa (`SEED`) garante os mesmos resultados em qualquer execução.

**Arquivos gerados pelo notebook:**

| Arquivo | O que é |
|---|---|
| `data/questionarios.csv` | a base simulada (passo 5) |
| `models/modelo.onnx` | o modelo treinado, no formato que o Java lê (passo 16) |
| `models/modelo_info.json` | o contrato: tudo o que o Java precisa saber para usar o modelo (passo 17) |
| `integracao_java/golden_tests.csv` | casos de teste com as respostas esperadas, para conferir o Java (passo 17) |

## 1. Bibliotecas
Todos os imports ficam aqui, no topo, como recomenda a PEP 8 (guia de estilo do Python): quem abre o notebook vê de cara todas as dependências. Ordem: biblioteca padrão, depois bibliotecas de terceiros.

| Biblioteca | Para que serve no projeto |
|---|---|
| `json`, `pathlib` | gravar o contrato em JSON e montar caminhos de arquivos |
| `numpy` | contas com vetores e matrizes (a simulação inteira é feita com ele) |
| `pandas` | tabelas: ler e gravar CSV, filtrar, agrupar, calcular médias |
| `matplotlib` | gráficos da análise exploratória e da avaliação |
| `scikit-learn` | modelos, validação cruzada, busca de hiperparâmetros e métricas |
| `skl2onnx` | converter o modelo do scikit-learn para o formato ONNX |
| `onnxruntime` | executar o arquivo ONNX, para conferir que ele reproduz o modelo |

In [ ]:
# ---- Biblioteca padrão do Python (já vem instalada) ----
# json: lê e grava arquivos JSON (texto chave-valor), usado no contrato com o Java (passo 17).
import json
# Path: representa caminhos de arquivos de forma independente do sistema operacional.
from pathlib import Path

# ---- Bibliotecas de terceiros (instaladas pelo requirements.txt) ----
# matplotlib.pyplot: desenha gráficos; "plt" é o apelido convencional.
import matplotlib.pyplot as plt
# numpy: vetores e matrizes numéricas com operações rápidas; "np" é o apelido convencional.
import numpy as np
# onnxruntime: executa modelos no formato ONNX, o mesmo motor que o Java usará.
import onnxruntime as ort
# pandas: tabelas (DataFrame) com colunas nomeadas, leitura e escrita de CSV; apelido "pd".
import pandas as pd
# to_onnx: converte um modelo do scikit-learn para o formato ONNX.
from skl2onnx import to_onnx
# FloatTensorType: descreve o tipo e o formato da entrada do modelo ONNX (float32, [N, 13]).
from skl2onnx.common.data_types import FloatTensorType
# clone: cria uma cópia "zerada" (sem treino) de um modelo, com os mesmos hiperparâmetros.
from sklearn.base import clone
# DummyClassifier: modelo "bobo" que ignora as features; serve de régua mínima (baseline).
from sklearn.dummy import DummyClassifier
# Dois modelos de árvores: Gradient Boosting (árvores em sequência) e Random Forest (árvores em paralelo).
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
# LogisticRegression: modelo linear que devolve probabilidades (o modelo escolhido no passo 10).
from sklearn.linear_model import LogisticRegression
# Métricas de avaliação: relatório por classe, F1, precisão e recall.
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
# Ferramentas de validação: busca em grade, divisão por grupos, previsões e avaliação por validação cruzada.
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_val_predict, cross_validate
# OneVsRestClassifier: treina um classificador binário por rótulo (aqui, um por conteúdo).
from sklearn.multiclass import OneVsRestClassifier
# make_pipeline: encadeia etapas (padronizar → classificar) num único objeto treinável.
from sklearn.pipeline import make_pipeline
# StandardScaler: padroniza cada feature para média 0 e desvio-padrão 1.
from sklearn.preprocessing import StandardScaler

## 2. Decisões do problema em código
As decisões do texto acima viram constantes: as séries atendidas, as disciplinas com os seus conteúdos (que definem as 8 saídas do modelo e a ordem delas), o valor que representa uma lacuna e a semente de reprodutibilidade.

**Conceitos usados:**
- **Constantes em MAIÚSCULAS:** convenção do Python (PEP 8) para valores que não devem mudar durante a execução.
- **Tupla × lista:** a tupla `( )` não pode ser alterada depois de criada; por isso é usada para dados fixos, como as séries.
- **Fonte única da verdade:** a tabela `CONTEUDOS_POR_DISCIPLINA` é escrita uma vez; `DISCIPLINAS`, `CONTEUDOS` e `DISCIPLINA_POR_CONTEUDO` são **derivados** dela. Assim, não há duas listas que possam ficar diferentes uma da outra.
- **Semente (seed):** os números "aleatórios" do computador são pseudoaleatórios, ou seja, gerados por uma conta a partir de um valor inicial. Com a mesma semente, a sequência é sempre a mesma, e o notebook dá sempre os mesmos resultados.
- **`assert`:** confere uma condição e para o notebook se ela for falsa. Aqui protege contra erros de digitação nas próprias constantes.

In [ ]:
# Decisões do problema em forma de código

# Séries atendidas: Ensino Fundamental I. "Série" é o nome usado na descrição da
# tarefa; na nomenclatura atual (BNCC) equivale a "ano" (1º ao 5º ano).
# Tupla (parênteses): sequência imutável, não pode ser alterada por engano depois.
SERIES = (1, 2, 3, 4, 5)

# Disciplinas e os seus conteúdos, inspirados na BNCC e nas matrizes do SAEB.
# A ordem é a ordem das 8 saídas do modelo (uma por conteúdo) e não deve mudar
# depois que o modelo for exportado: o Java depende dela.
# Dicionário {chave: valor}: cada disciplina (chave) aponta para a tupla dos seus conteúdos (valor).
CONTEUDOS_POR_DISCIPLINA = {
    # Língua Portuguesa: ler a palavra, achar a informação, entender o implícito, escrever certo.
    "portugues":  ("decodificacao", "localizar_informacao", "inferencia", "ortografia"),
    # Matemática: sistema de numeração, contas, formas e espaço, medidas.
    "matematica": ("numeros", "operacoes", "geometria", "grandezas_medidas"),
}

# Atalhos derivados (nunca digitados à mão)
# tuple(dicionário) pega só as chaves, na ordem em que foram escritas: ("portugues", "matematica").
DISCIPLINAS = tuple(CONTEUDOS_POR_DISCIPLINA)
# Expressão geradora com dois "for": percorre cada tupla de conteúdos e depois cada conteúdo dela,
# achatando tudo numa única tupla com os 8 conteúdos.
CONTEUDOS = tuple(c for conteudos in CONTEUDOS_POR_DISCIPLINA.values() for c in conteudos)
# Compreensão de dicionário: o caminho inverso, de cada conteúdo para a sua disciplina.
DISCIPLINA_POR_CONTEUDO = {c: d for d, conteudos in CONTEUDOS_POR_DISCIPLINA.items() for c in conteudos}

# Rótulo de cada conteúdo: 1 = lacuna (abaixo do esperado para a série), 0 = sem lacuna.
# A lacuna é a classe positiva: a métrica prioritária é o recall dela.
LACUNA = 1

# Semente fixa: torna reproduzível toda etapa aleatória.
# Com a mesma semente, o gerador de números aleatórios produz sempre a mesma sequência.
SEED = 42

# Validações: interrompem o notebook se houver erro de digitação.
# assert condição, mensagem → se a condição for falsa, o Python para com AssertionError e a mensagem.
# sorted(set(...)) ordena e remove repetidos; se o resultado for igual ao original, já estava em ordem e sem repetição.
assert list(SERIES) == sorted(set(SERIES)), "SERIES deve estar em ordem e sem repetição"
# set() descarta repetidos: se o tamanho cai, algum nome de conteúdo aparece duas vezes.
assert len(CONTEUDOS) == len(set(CONTEUDOS)), "há conteúdo com nome repetido entre disciplinas"
# f-string: o texto entre chaves {} é substituído pelo valor da expressão.
print(f"{len(SERIES)} séries, {len(DISCIPLINAS)} disciplinas, {len(CONTEUDOS)} conteúdos: {CONTEUDOS}")